# TopoForge x GeNN: segregated vs interleaved placement on an independent engine## READ THIS FIRST - THIS NOTEBOOK IS UNTESTED**It has never been executed.** It was written on a machine with no GPU, no CUDAand no C++ compiler, so GeNN could not be installed, let alone run. Every GeNNAPI call below was written by reading real source, not from memory, but *readingsource is not running code*. Expect to debug this on first run. Nothing in thisnotebook reports a result; there are no placeholder numbers and no example output.### What was verified against real sourceFetched 2026-09-02. "Reference" below means[`jhnnsnk/genn_structural_plasticity`](https://github.com/jhnnsnk/genn_structural_plasticity)(MIT), the code accompanying Knight, Senk & Nowotny (2026), *A flexible frameworkfor structural plasticity in GPU-accelerated sparse spiking neural networks*,Neuromorphic Computing and Engineering 6(1) 014019, doi:10.1088/2634-4386/ae4535."PyGeNN source" means `genn-team/genn` at `master`.| Thing | Status | Where it came from ||---|---|---|| Colab install cell (`gdown` + `pygenn-5.4.0-cp313...whl` + `CUDA_PATH`) | **verified** | copied from the GeNN team's own tutorial notebooks: `genn-team/genn` `docs/tutorials/mushroom_body/*.ipynb` and `genn-team/ml_genn` `docs/tutorials/*.ipynb` || Source-build fallback command | **verified** | `genn-team/genn` `docs/installation.rst` || `BamfordStructuralPlasticity` custom connectivity update model | **verified, copied verbatim** | reference `topomap/custom_models_and_snippets.py` || `GaussianProfileWithoutReplacement` connectivity snippet | **verified, copied verbatim except the row/col-length caps** | same file || `STDPAllToAllSpikePairing` weight update model | **verified, copied verbatim** | same file || `PoissonSpatiallyCorrelatedInput` neuron model | **verified, copied verbatim** | same file || `add_neuron_population` / `add_synapse_population` / `add_custom_connectivity_update` signatures | **verified** | PyGeNN source `pygenn/genn_model.py` (checked against the reference's usage; no drift between 5.3 and master) || `init_weight_update` / `init_postsynaptic` / `init_sparse_connectivity` / `init_var` / `create_var_ref` / `create_wu_var_ref` | **verified** | reference `topomap/parameters.py` + `topographic_map_model.py`; exports confirmed in `pygenn/__init__.py` || Connectivity readout: `pull_connectivity_from_device()`, `get_sparse_pre_inds()`, `get_sparse_post_inds()` | **verified** | reference `topographic_map_model.get_connectivity`; implementation read in `pygenn/genn_groups.py` || Weight readout: `sg.vars["g"].pull_from_device(); .values` returns an array element-aligned with the sparse pre/post inds | **verified** | `pygenn/model_preprocessor.py` `SynapseVariable.values` - both it and `get_sparse_post_inds` hstack rows using the same `_row_lengths` || Neuron var write: `var.pull_from_device(); var.values = arr; var.push_to_device()` | **verified** | reference `topographic_map_model.set_stimulus_rates` || EGP init: `splast.extra_global_params[...].set_init_values(np.zeros(...))` | **verified** | reference `topographic_map_model.add_connectivity_update` || `model.custom_update("UpdateConnectivity")`, `model.step_time()`, `model.timestep`, `model.t`, `model.seed`, `model.dt`, `model.build()`, `model.load()`, `model.unload()` | **verified** | reference `run_experiments.run_model` + `pygenn/genn_model.py` || Built-in `LIF` sim/threshold/reset code and derived params | **verified** | GeNN C++ source `include/genn/genn/neuronModels.h` || `LIFWithSpikeCount` (our transcription of that LIF plus a `spikeCount` var) | **inferred** | mechanical transcription; the added `spikeCount++` in `reset_code` is ours and is not in any published source || Whether the parameter values below produce a network that actually fires and actually settles | **NOT verified - this is the main risk** | see the smoke-test cell, which exists to check exactly this || Wall-clock runtime on a Colab GPU | **unknown** | time the smoke test and extrapolate |### Known traps- `pip install pygenn` and `pip install genn` from PyPI are **unrelated packages  by other authors**. GeNN is not on PyPI. Use the wheel or the GitHub archive.- The prebuilt wheel is `cp313` / `linux_x86_64`. If Colab is not on Python 3.13  it will refuse to install and the notebook falls back to a source build  (several minutes).- The wheel is GeNN **5.4.0**; the reference `topomap` code was tested on  **5.3.0**. The API surface used here is unchanged between them (checked against  `master`), but if something breaks, pinning 5.3.0 is the first thing to try.

## 1. Install GeNN

In [ ]:
# VERIFIED: this is the install cell used by the GeNN team's own tutorial
# notebooks (genn-team/genn docs/tutorials/mushroom_body/*.ipynb and
# genn-team/ml_genn docs/tutorials/*.ipynb), which is verbatim:
#
#     if "google.colab" in str(get_ipython()):
#         !gdown 19X8ndTSeyQyTHPBLqllMd9JensovY6ce
#         !pip install pygenn-5.4.0-cp313-cp313-linux_x86_64.whl
#         %env CUDA_PATH=/usr/local/cuda
#
# DO NOT use `pip install pygenn` or `pip install genn` - those are unrelated
# PyPI packages by other authors. GeNN is not distributed on PyPI.
import sys

IN_COLAB = "google.colab" in str(get_ipython())
print("Python:", sys.version)

if IN_COLAB:
    !nvidia-smi -L || echo "NO GPU DETECTED - see the CPU fallback note below"
    %env CUDA_PATH=/usr/local/cuda
    if sys.version_info[:2] == (3, 13):
        !gdown 19X8ndTSeyQyTHPBLqllMd9JensovY6ce
        !pip install -q pygenn-5.4.0-cp313-cp313-linux_x86_64.whl
    else:
        # Fallback, VERIFIED against genn-team/genn docs/installation.rst.
        # 5.3.0 is the version the reference topomap code was tested with.
        print("Colab Python is not 3.13, so the prebuilt cp313 wheel will not")
        print("install. Building GeNN 5.3.0 from source - this takes minutes.")
        !apt-get -qq install -y libffi-dev
        !pip install -q -U pip
        !pip install -q https://github.com/genn-team/genn/archive/refs/tags/5.3.0.zip
else:
    print("Not on Colab - assuming pygenn is already installed.")

In [ ]:
import pygenn
print("pygenn version:", pygenn.__version__)

## 2. What this experiment does, and what it does notTopoForge's claim is that on plastic neuromorphic hardware, *where you placeneurons* changes what the network can wire up - not just how much energy it coststo communicate. The paper's Limitations section flags a real weakness: everynumber in it comes from one simulator, written by one person. This notebookaddresses that one weakness and nothing else.**Template.** `src/experiments/exp50_published_rule_confirmatory.py` in theTopoForge repo. That experiment runs a *published, value-free, type-blind*structural plasticity rule (Butz & van Ooyen 2013) over two placements thatdiffer **only** in which neuron type sits at which fixed position, and scores the**cross-type fraction of grown synapses** against a **type-blind chance ceiling**.This notebook is the same design, run on GeNN with a different published rule(Bamford-style structural plasticity as implemented by Knight, Senk & Nowotny).**The framing point, which is easy to get backwards.** With `NC` balanced types,a type-blind partner draw gives an expected cross-type fraction of`(N - N/NC) / (N - 1)` - here about **0.801**. The interleaved condition sits*at* that ceiling and **cannot exceed it**. So the finding is *not* thatinterleaving boosts cross-type wiring. It is that **segregation suppressescross-type wiring far below what type-blind chance would produce**, becausesame-type blocks fill each neuron's spatial neighbourhood and the formationkernel is local.**Scope, stated plainly.**1. This is a **same-rule-family replication on an independent engine**. It tests   whether a third-party GPU simulator, running a third party's implementation of   a published distance-dependent formation rule, reproduces the geometric effect.2. It is **not a hardware result** and does not close the hardware gap.3. The Bamford formation rule is *purely* distance-dependent, so the cross-type   fraction here is largely predicted by geometry alone - the notebook computes   that analytic prediction so you can see how much of the effect is kernel   geometry and how much is activity-dependent retention via STDP. Do not read   this as an independent test of the *learning* claim. It is a test of the   placement-to-connectivity step.**Deviations from the reference, all deliberate and all listed:**- `dt = 1.0 ms` rather than the reference's `0.1 ms`, for Colab runtime. The  reference's own comment notes that Bogdan et al. 2018, the model it follows,  uses 1 ms. Set `DT = 0.1` to match the reference exactly if you have the time.- Initial lateral connectivity is made **almost empty** (`INIT_PEAK_PROB`  ~ 0.001) so that essentially all measured connectivity is *grown by the  plasticity rule*, as in exp50. The reference starts from a Gaussian-initialised  network instead.- `NUM_REWIRING_ATTEMPTS` is raised well above the reference's per-neuron rate so  the network reaches steady state inside a Colab-length run. This changes the  rewiring **rate**, not the rule. The settling check exists to verify it worked.- Row/column length caps in the connectivity snippet are made configurable  (the reference hardcodes `2 * 32` and `32`).- The neuron grid is 30x30 rather than the reference's 16x16, to hit N = 900 and  match exp50.

## 3. Configuration

In [ ]:
import time
import numpy as np
from scipy import stats

# ---------------------------------------------------------------------------
# Geometry. Positions are IMPLICIT IN THE NEURON INDEX, exactly as in the
# reference: its distance code computes x = id % grid_num_x, y = id / grid_num_x.
# That is what makes the core invariant of this experiment free: both conditions
# use one and the same coordinate array, because they use the same index set.
GRID = 30                     # 30 x 30
N = GRID * GRID               # 900 neurons, matching exp50
NC = 5                        # number of neuron types
assert GRID % NC == 0, "segregated stripes must divide the grid evenly"
STRIPE_H = GRID // NC         # 6 rows per type in the segregated condition

PLACEMENT_SEED = 42           # exp32b uses 42 for its placement RNG

# Input pattern schedule, copied from exp32b_benchmark.PATTERNS.
# Types 0 and 3 co-activate; types 1 and 4 co-activate; type 2 alone.
PATTERNS = [(0, 3), (1, 4), (2,)]
PATTERN_BLOCK_MS = 100.0
BASE_RATE = 5.0               # Hz, reference corr_base_rate
PEAK_RATE = 200.0             # Hz - NOT calibrated, see the smoke test

# ---------------------------------------------------------------------------
# Simulation timing. DEVIATION: reference parameters.py uses dt = 0.1 ms.
DT = 1.0                      # ms
TSIM_MS = 20000.0             # ms of biological time per run
UPDATE_INTERVAL_MS = 1.0      # reference value: rewire once per ms

# ---------------------------------------------------------------------------
# LIF target neurons - parameter values copied verbatim from reference
# parameters.py "neuron_param_space_tgt".
LIF_PARAMS = {"C": 20.0, "TauM": 20.0, "Vrest": -70.0, "Vreset": -70.0,
              "Vthresh": -54.0, "Ioffset": 0.0, "TauRefrac": 5.0}
PS_PARAMS = {"tau": 5.0, "E": 0.0}          # reference "ps_param_space"

FF_WEIGHT = 0.05              # source -> target one-to-one. NOT calibrated.

# ---------------------------------------------------------------------------
# STDP - copied verbatim from reference parameters.py.
G_MAX = 0.2
G_INIT = 0.2
TAU_PLUS = 20.0
TAU_MINUS = 64.0
A_PLUS = 0.1 * 0.2
B = 1.2
A_MINUS = B * A_PLUS * TAU_PLUS / TAU_MINUS

# ---------------------------------------------------------------------------
# Lateral connectivity kernel - reference values for the lateral population.
PEAK_PROB_LAT = 1.0
STD_LAT = 1.0                 # in units of grid spacing
INIT_PEAK_PROB = 0.001        # DEVIATION: start (almost) empty, as exp50 does
MAX_ROW_LEN = 128             # DEVIATION: reference hardcodes 2 * 32
MAX_COL_LEN = 128             # DEVIATION: reference hardcodes 32

# ---------------------------------------------------------------------------
# Structural plasticity - reference values except the attempt rate.
WEIGHT_THRESHOLD = 0.1                 # 0.5 * g_max
NEW_WEIGHT_INIT = 0.2                  # g_max
ELIM_DEP = 0.0245 * 50                 # reference value (note: > 1, so a
ELIM_POT = 0.000136 * 50               # selected depressed synapse always goes)
# DEVIATION: the reference uses 10 attempts/ms for 256 neurons, i.e. 35 for 900.
# We use 10x that so the network reaches steady state inside a Colab-length run.
NUM_REWIRING_ATTEMPTS = 350

BITFIELD_ROW_WORDS = int((N + 31) / 32)  # reference formula

# ---------------------------------------------------------------------------
# Seeds. exp50's seeds, so the two experiments are directly comparable.
SEEDS = list(range(100, 112))            # 12 seeds per condition

# ---------------------------------------------------------------------------
# PRE-REGISTERED thresholds. These are NOT chosen after looking at any GeNN
# output - they are exp50's registered values, lifted unchanged.
SEG_CEILING_MAX = 0.60        # P2b: segregated must be suppressed below this
INT_CEILING_MIN = 0.70        # P2c: interleaved must sit at/near the ceiling
SETTLE_TOL = 0.10             # P1b: edge count must change < 10% in last quarter

print("N = {}, NC = {}, {} neurons per type".format(N, NC, N // NC))
print("{} ms per run at dt = {} ms -> {} timesteps".format(
    TSIM_MS, DT, int(round(TSIM_MS / DT))))
print("{} seeds x 2 conditions = {} runs".format(len(SEEDS), 2 * len(SEEDS)))

## 4. The two placements, the invariant, and the chance ceilingEverything in this section is plain NumPy. It runs whether or not GeNN installed,so run it first - if the invariant assertions fail, nothing downstream is worthdebugging.

In [ ]:
# Coordinates. VERIFIED to match the reference's C code, which computes
#     xPre = id_pre % grid_num_x;  yPre = id_pre / grid_num_x;
# so neuron index i sits at (i % GRID, i // GRID).
IDX = np.arange(N)
XPOS = (IDX % GRID).astype(float)
YPOS = (IDX // GRID).astype(float)
COORDS = np.stack([XPOS, YPOS], axis=1)

# --- SEGREGATED: each type occupies one contiguous horizontal stripe.
# The analogue of exp32b's make_placement("vlsi").
types_seg = (YPOS // STRIPE_H).astype(int)

# --- INTERLEAVED: the SAME multiset of types, permuted across the SAME
# positions. The analogue of exp32b's make_placement("topoforge").
_rng = np.random.default_rng(PLACEMENT_SEED)
types_int = types_seg.copy()
_rng.shuffle(types_int)

CONDITIONS = {"segregated": types_seg, "interleaved": types_int}

# --- P1a: THE INVARIANT. The two conditions differ ONLY in the index -> type
# assignment. Coordinates are not merely equal, they are the same object, so
# every spatial quantity the rule can see is exactly equal by construction.
print("[P1a] structural parity")
print("  one shared coordinate array:", COORDS.shape)
assert np.array_equal(np.sort(types_seg), np.sort(types_int)), \
    "type multisets differ - conditions are not adjacency-matched"
assert np.array_equal(np.bincount(types_seg, minlength=NC),
                      np.bincount(types_int, minlength=NC)), \
    "per-type counts differ"
assert not np.array_equal(types_seg, types_int), \
    "the two conditions are identical - nothing is being compared"
print("  per-type counts segregated :", np.bincount(types_seg, minlength=NC).tolist())
print("  per-type counts interleaved:", np.bincount(types_int, minlength=NC).tolist())
print("  multisets identical, assignments differ -> P1a HOLDS")


def chance_cross_fraction(types):
    """Expected cross-type fraction for a TYPE-BLIND random partner.
    Copied from exp50_published_rule_confirmatory.chance_cross_fraction."""
    counts = np.bincount(types, minlength=NC).astype(float)
    per_neuron = (len(types) - counts) / (len(types) - 1.0)
    return float((counts * per_neuron).sum() / len(types))


CEILING = chance_cross_fraction(types_int)
print("\n  type-blind chance cross-type fraction (the CEILING): {:.4f}".format(CEILING))
print("  interleaved is expected to sit AT this and cannot exceed it;")
print("  the claim under test is that segregated is pushed far BELOW it.")

In [ ]:
# Analytic prediction from the formation kernel alone.
#
# The reference's formation rule samples a candidate postsynaptic index
# uniformly and accepts it with probability peak_prob * exp(-dd / (2*std^2)),
# with dd computed under periodic boundary conditions. So, ignoring elimination,
# the distribution of formed synapses over pairs (i, j) is proportional to
# gauss(d_ij). This cell computes the resulting cross-type fraction exactly.
#
# It is a prediction, not a result: it tells you what geometry ALONE implies,
# so that when GeNN produces a number you can see how far activity-dependent
# retention moved it.

def _pbc_delta(a, b, L):
    """Mirrors the reference C code:
         d = fabs(post - pre); if (d > 0.5*L) d = d - L;
       which is the minimum-image displacement (sign is irrelevant, it is squared)."""
    d = np.abs(a[:, None] - b[None, :])
    return np.where(d > 0.5 * L, d - L, d)


DX = _pbc_delta(XPOS, XPOS, GRID)
DY = _pbc_delta(YPOS, YPOS, GRID)
DD = DX * DX + DY * DY
KERNEL = PEAK_PROB_LAT * np.exp(-DD / (2.0 * STD_LAT * STD_LAT))
np.fill_diagonal(KERNEL, 0.0)          # we exclude autapses in the readout too


def kernel_cross_fraction(types):
    diff = (types[:, None] != types[None, :])
    return float((KERNEL * diff).sum() / KERNEL.sum())


print("Analytic cross-type fraction implied by the formation kernel alone")
print("(no elimination, no STDP, no simulation):")
for name, types in CONDITIONS.items():
    print("  {:<12} {:.4f}".format(name, kernel_cross_fraction(types)))
print("  {:<12} {:.4f}  <- type-blind chance ceiling".format("ceiling", CEILING))
print()
print("Mean kernel mass per neuron (expected in-degree scale): {:.2f}".format(
    KERNEL.sum(axis=1).mean()))

## 5. ModelsEverything in the next cell except `LIFWithSpikeCount` is **copied verbatim** from`topomap/custom_models_and_snippets.py` in the reference repository, which isMIT-licensed. The only edits are the two `calc_max_*_len_func` lambdas, changedfrom hardcoded `2 * 32` / `32` to our configurable caps, and the comment markingthat edit.`LIFWithSpikeCount` is **ours**: GeNN's built-in `LIF` (transcribed from`include/genn/genn/neuronModels.h`) plus a `spikeCount` variable incremented in`reset_code`. We add it so that firing rates can be read back through thealready-verified `vars[...].pull_from_device(); .values` path rather than through`spike_recording_data`, whose return shape we could not verify.

In [ ]:
from string import Template

from pygenn import (GeNNModel,
                    create_custom_connectivity_update_model,
                    create_neuron_model,
                    create_sparse_connect_init_snippet,
                    create_weight_update_model,
                    create_var_ref, create_wu_var_ref,
                    init_postsynaptic, init_sparse_connectivity,
                    init_var, init_weight_update)

# =========================================================================
# BEGIN verbatim copy from jhnnsnk/genn_structural_plasticity (MIT licence),
# topomap/custom_models_and_snippets.py
# =========================================================================

distance_squared = Template("""
    // x- and y-positions
    const scalar xPre = $id_pre % grid_num_x;
    const scalar xPost = $id_post % grid_num_x;
    const scalar yPre = $id_pre / grid_num_x;
    const scalar yPost = $id_post / grid_num_x;

    // Absolute displacement in x- and y-direction
    scalar dx = fabs( xPost - xPre );
    scalar dy = fabs( yPost - yPre );

    // Periodic boundary conditions
    if (dx > 0.5 * grid_num_x){
        dx = dx - grid_num_x;
    };
    if (dy > 0.5 * grid_num_y){
        dy = dy - grid_num_y;
    };

    // Squared distance
    const scalar dd = dx * dx + dy * dy;
    """)

PoissonSpatiallyCorrelatedInput = create_neuron_model(
    "PoissonSpatiallyCorrelatedInput",
    params=["stim_interval"],
    vars=[("timeStepToSpike", "scalar"), ("stim_rates", "scalar")],
    sim_code="""
        // Reset timeStepToSpike after each stimulus interval such that the
        // new stimulus location immediately becomes effective
        if ( (int) (t / dt) % (int) (stim_interval / dt) == 0){
            timeStepToSpike = 0.;
        }

        if(timeStepToSpike <= 0.0) {
            const scalar isi = 1000. / (stim_rates * dt);
            timeStepToSpike += isi * gennrand_exponential();
        }
        timeStepToSpike -= 1.0;
        """,
    threshold_condition_code="""
        timeStepToSpike <= 0.0
        """
)

GaussianProfileWithoutReplacement = \
    create_sparse_connect_init_snippet(
        "GaussianProfileWithoutReplacement",
        params=[
            "peak_prob", "std",
            ("grid_num_x", "unsigned int"), ("grid_num_y", "unsigned int")],
        col_build_code="""
            // iterate over all potential source neurons
            for (unsigned int idPre=0; idPre<num_pre; idPre++) {
        """
        + distance_squared.substitute(id_pre="idPre", id_post="id_post") +
        """
                // Gaussian spatial profile
                const scalar gauss = peak_prob * exp(-dd / ( 2.0 * std * std));

                // Establish synapse if random number is smaller than Gaussian
                // profile at given distance
                if (gennrand_uniform() < gauss)
                {
                    addSynapse(idPre + id_pre_begin);
                }
            }
            """,
        # EDIT vs reference: reference hardcodes `2 * 32` and `32` here. We make
        # the caps configurable because structural plasticity must have room to
        # grow rows; hitting the cap would silently drop synapses.
        calc_max_row_len_func=lambda num_pre, num_post, pars: MAX_ROW_LEN,
        calc_max_col_len_func=lambda num_pre, num_post, pars: MAX_COL_LEN)

STDPAllToAllSpikePairing = create_weight_update_model(
    class_name="STDPAllToAllSpikePairing",
    params=["tauPlus", "tauMinus", "aPlus", "aMinus", "wMin", "wMax"],
    derived_params=[
        ("tauPlusDecay", lambda pars, dt: np.exp(-dt / pars["tauPlus"])),
        ("tauMinusDecay", lambda pars, dt: np.exp(-dt / pars["tauMinus"]))
    ],
    vars=[("g", "scalar")],
    pre_vars=[("preTrace", "scalar")],
    post_vars=[("postTrace", "scalar")],

    pre_spike_syn_code="""
        addToPost(g);
        const scalar tDiff = t - st_post;
        if(tDiff > 0) {
            const scalar newWeight = g - (aMinus * postTrace);
            g = fmin(wMax, fmax(wMin, newWeight));
        }
        """,
    post_spike_syn_code="""
        const scalar tDiff = t - st_pre;
        if(tDiff > 0) {
            const scalar newWeight = g + (aPlus * preTrace);
            g = fmin(wMax, fmax(wMin, newWeight));
        }
        """,
    pre_spike_code="""
        preTrace += 1.0;
        """,
    pre_dynamics_code="""
        preTrace *= tauPlusDecay;
        """,
    post_spike_code="""
        postTrace += 1.0;
        """,
    post_dynamics_code="""
        postTrace *= tauMinusDecay;
        """,
)

BamfordStructuralPlasticity = \
    create_custom_connectivity_update_model(
        class_name="BamfordStructuralPlasticity",
        params=[("num_rewiring_attempts", "unsigned int"),
                "weight_threshold",
                "new_weight_init",
                "peak_prob",
                "std",
                ("grid_num_x", "unsigned int"),
                ("grid_num_y", "unsigned int"),
                "elimination_probability_dep",
                "elimination_probability_pot"],
        extra_global_params=[("num_rewiring_attempts_per_row", "unsigned int*"),
                             ("bitfield", "uint32_t*")
                             ],
        var_refs=[("g", "scalar")],
        host_update_code="""
            // Set the number of rewiring attempts per presynaptic neuron to zero
            for(unsigned int i = 0; i < num_pre; i++) {
                num_rewiring_attempts_per_row[i] = 0;
            }

            // Distribute the rewiring attempts uniformly over the presynaptic neurons.
            // gennrand() samples a random 32-bit unsigned integer and the
            // modulo operation restricts it to [0, num_pre).
            // Note: Multiple attempts for the same idPre are possible, up to
            // the total number of postsynaptic neurons.
            for (unsigned int i = 0; i < num_rewiring_attempts; i++){
                unsigned int idPre;
                bool num_post_not_reached = true;
                while (num_post_not_reached == true) {
                idPre = gennrand() % num_pre;
                    if (num_rewiring_attempts_per_row[idPre] < num_post){
                        num_rewiring_attempts_per_row[idPre]++;
                        num_post_not_reached = false;
                    }
                }
            }

            // Push to device
            pushnum_rewiring_attempts_per_rowToDevice(num_pre);
            """,
        row_update_code="""
            // Code for each presynaptic neuron
            // (id_pre and id_post (in for_each_synapse{}) are set elsewhere)
            // Sample (without replacement) postsynaptic indices for rewiring
            // attempts

            // Number of rewiring attempts of this row
            const unsigned int n = num_rewiring_attempts_per_row[id_pre];

            // Identify section of bitfield this presynaptic neuron operates on
            const unsigned int bitfieldRowWords = (num_post + 31) / 32;
            uint32_t *bitfieldRow = bitfield + (bitfieldRowWords * id_pre);

            // Number of idPosts visited
            unsigned int t = 0;

            // Number of idPosts selected so far
            for (unsigned int m = 0; m < n;) {
                // Sample uniform random number in [0.0, 1.0]
                if ((num_post - t) * gennrand_uniform() >= n - m){
                    t++;
                }
                else{
                    const unsigned int word = t / 32;
                    const unsigned int bit = 1 << (t % 32);
                    bitfieldRow[word] |= bit;

                    t++;
                    m++;
                }
            }

            // Elimination rule:
            // Iterate over all existing synapses of this presynaptic neuron.
            // If a synapse has been selected, it gets eliminated with a given
            // probability.
            for_each_synapse {
                // Check if bit is set
                const unsigned int word = id_post / 32;
                const unsigned int bit = 1 << (id_post % 32);
                if ( bitfieldRow[word] & bit ){

                    // Check if weight is below threshold
                    const scalar elimProb =  ((g < weight_threshold)
                                              ? elimination_probability_dep
                                              : elimination_probability_pot);

                    // Remove dependent on elimination probability
                    if ( gennrand_uniform() < elimProb ){
                        remove_synapse();
                    }
                    // Clear bit
                    bitfieldRow[word] &= ~bit;
                }
            }

            // Formation rule:
            // Iterate over bitfield row of this presynaptic neuron.
            // If a bit it set, a synapse to the corresponding postsynaptic neuron
            // is created.
            const unsigned int numPostWords =  (num_post + 31) / 32;
            for ( unsigned int word = 0; word < numPostWords; word++ ) {
                if ( bitfieldRow[word] != 0 ) {
                    for ( unsigned int bit = 0; bit < 32; bit++ ) {
                        // Check if bit is set
                        if ( bitfieldRow[word] & (1 << bit) ){
                            const unsigned int idPost = (word * 32) + bit;
                            """
        + distance_squared.substitute(id_pre="id_pre", id_post="idPost") +
        """
                            // Gaussian spatial profile
                            const scalar gauss = peak_prob * exp(-dd / (2.0 * std * std));

                            // Add synapse with distance-dependent probability
                            if ( gennrand_uniform() < gauss){
                                add_synapse(idPost, new_weight_init);
                            }

                            // Clear bit
                            bitfieldRow[word] &=  ~(1 << bit);
                        }
                    }
                }
            }
            """
    )

# =========================================================================
# END verbatim copy
# =========================================================================

# OURS, INFERRED. GeNN's built-in LIF, transcribed from
# include/genn/genn/neuronModels.h (class LIF), plus a spikeCount var so
# per-neuron firing rates can be read back through the verified variable path.
LIFWithSpikeCount = create_neuron_model(
    "LIFWithSpikeCount",
    params=["C", "TauM", "Vrest", "Vreset", "Vthresh", "Ioffset", "TauRefrac"],
    vars=[("V", "scalar"), ("RefracTime", "scalar"), ("spikeCount", "scalar")],
    derived_params=[
        ("ExpTC", lambda pars, dt: np.exp(-dt / pars["TauM"])),
        ("Rmembrane", lambda pars, dt: pars["TauM"] / pars["C"])],
    sim_code="""
        if (RefracTime <= 0.0) {
          scalar alpha = ((Isyn + Ioffset) * Rmembrane) + Vrest;
          V = alpha - (ExpTC * (alpha - V));
        }
        else {
          RefracTime -= dt;
        }
        """,
    threshold_condition_code="RefracTime <= 0.0 && V >= Vthresh",
    reset_code="""
        V = Vreset;
        RefracTime = TauRefrac;
        spikeCount += 1.0;
        """,
    auto_refractory_required=False)

print("models defined")

## 6. Building and running one conditionNote what the GeNN model does **not** contain: any notion of neuron *type*. Typesenter in exactly two places, both outside the simulator - the host-side stimulusschedule (which types are driven in which block) and the Python-side readout. Theformation and elimination rules are type-blind by construction, not by assertion.

In [ ]:
def build_model(seed, model_name="TopoForgeGeNN"):
    """Build the model. Returns (model, ng_src, ng_tgt, sg_lat)."""
    model = GeNNModel(model_name=model_name)
    model.dt = DT
    model.seed = int(seed)

    ng_src = model.add_neuron_population(
        pop_name="SourceLayer", num_neurons=N,
        neuron=PoissonSpatiallyCorrelatedInput,
        params={"stim_interval": PATTERN_BLOCK_MS},
        vars={"timeStepToSpike": 0.0, "stim_rates": BASE_RATE})

    ng_tgt = model.add_neuron_population(
        pop_name="TargetLayer", num_neurons=N,
        neuron=LIFWithSpikeCount,
        params=LIF_PARAMS,
        vars={"V": init_var("Normal", {"mean": -60.0, "sd": 5.0}),
              "RefracTime": 0.0,
              "spikeCount": 0.0})

    # Feedforward drive: one source Poisson neuron per target neuron, static
    # weight, no structural plasticity. This is the only channel through which
    # a neuron's TYPE reaches the simulation, via its stim_rates.
    model.add_synapse_population(
        pop_name="FeedForward", matrix_type="SPARSE",
        source=ng_src, target=ng_tgt,
        weight_update_init=init_weight_update(snippet="StaticPulse",
                                              vars={"g": FF_WEIGHT}),
        postsynaptic_init=init_postsynaptic(
            snippet="ExpCond", params=PS_PARAMS,
            var_refs={"V": create_var_ref(ng_tgt, "V")}),
        connectivity_init=init_sparse_connectivity(snippet="OneToOne", params={}))

    # Lateral recurrent population: this is the connectivity we measure.
    sg_lat = model.add_synapse_population(
        pop_name="Lateral", matrix_type="SPARSE",
        source=ng_tgt, target=ng_tgt,
        weight_update_init=init_weight_update(
            snippet=STDPAllToAllSpikePairing,
            params={"tauPlus": TAU_PLUS, "tauMinus": TAU_MINUS,
                    "aPlus": A_PLUS, "aMinus": A_MINUS,
                    "wMin": 0.0, "wMax": G_MAX},
            vars={"g": G_INIT},
            pre_vars={"preTrace": 0.0},
            post_vars={"postTrace": 0.0}),
        postsynaptic_init=init_postsynaptic(
            snippet="ExpCond", params=PS_PARAMS,
            var_refs={"V": create_var_ref(ng_tgt, "V")}),
        # DEVIATION: peak_prob is INIT_PEAK_PROB, not PEAK_PROB_LAT, so the
        # network starts essentially empty and the measured connectivity is
        # grown by the plasticity rule (as in exp50).
        connectivity_init=init_sparse_connectivity(
            snippet=GaussianProfileWithoutReplacement,
            params={"peak_prob": INIT_PEAK_PROB, "std": STD_LAT,
                    "grid_num_x": GRID, "grid_num_y": GRID}))

    splast = model.add_custom_connectivity_update(
        cu_name="LateralStructuralPlasticity",
        group_name="UpdateConnectivity",
        syn_group=sg_lat,
        custom_conn_update_model=BamfordStructuralPlasticity,
        params={"num_rewiring_attempts": NUM_REWIRING_ATTEMPTS,
                "weight_threshold": WEIGHT_THRESHOLD,
                "new_weight_init": NEW_WEIGHT_INIT,
                "peak_prob": PEAK_PROB_LAT,
                "std": STD_LAT,
                "grid_num_x": GRID,
                "grid_num_y": GRID,
                "elimination_probability_dep": ELIM_DEP,
                "elimination_probability_pot": ELIM_POT},
        var_refs={"g": create_wu_var_ref(sg_lat, "g")})

    # Reference passes np.zeros (float64) for these despite their declared
    # unsigned int* / uint32_t* types; mirrored verbatim rather than "fixed".
    splast.extra_global_params["num_rewiring_attempts_per_row"].set_init_values(
        np.zeros(N))
    splast.extra_global_params["bitfield"].set_init_values(
        np.zeros(BITFIELD_ROW_WORDS * N))

    return model, ng_src, ng_tgt, sg_lat


def set_stim_rates(rates_var, types, block):
    """Drive the types in the current pattern; everything else at base rate.
    Mirrors exp32b's schedule: PATTERNS[p] gets the boost, cycling over blocks.
    Write path copied from reference topographic_map_model.set_stimulus_rates."""
    p = block % len(PATTERNS)
    r = np.full(N, BASE_RATE, dtype=float)
    r[np.isin(types, PATTERNS[p])] = PEAK_RATE
    rates_var.pull_from_device()
    rates_var.values = r
    rates_var.push_to_device()


def read_connectivity(sg):
    """Copied from reference topographic_map_model.get_connectivity, plus
    weights. pull_connectivity_from_device MUST come first: both the index
    readout and the sparse `values` readout depend on the pulled row lengths."""
    sg.pull_connectivity_from_device()
    pre = np.asarray(sg.get_sparse_pre_inds())
    post = np.asarray(sg.get_sparse_post_inds())
    gvar = sg.vars["g"]
    gvar.pull_from_device()
    g = np.asarray(gvar.values)
    return pre, post, g

In [ ]:
def measure(pre, post, g, types):
    """Primary endpoint: cross-type fraction of grown synapses, autapses excluded.

    Autapses are excluded because the reference's formation rule does not forbid
    i -> i, and an autapse is trivially same-type; including them would depress
    both conditions by the same nuisance amount. The count is reported so the
    decision is visible rather than buried."""
    n_auto = int((pre == post).sum())
    keep = pre != post
    pre, post, g = pre[keep], post[keep], g[keep]
    n = len(pre)
    if n == 0:
        return dict(n_edges=0, n_autapses=n_auto, cross_frac=float("nan"),
                    cross_count=0, taught=0, mean_g=float("nan"),
                    n_pot=0, cross_frac_pot=float("nan"), max_row_len=0)

    tp, tq = types[pre], types[post]
    cross = tp != tq

    # Secondary, for continuity with exp32b: the two pattern-coupled type pairs.
    taught = int((((tp == 0) & (tq == 3)) | ((tp == 3) & (tq == 0)) |
                  ((tp == 1) & (tq == 4)) | ((tp == 4) & (tq == 1))).sum())

    # Activity-sensitive secondary endpoint: restrict to synapses STDP has
    # driven above the elimination threshold, i.e. the ones the rule protects.
    pot = g >= WEIGHT_THRESHOLD

    return dict(
        n_edges=n,
        n_autapses=n_auto,
        cross_count=int(cross.sum()),
        cross_frac=float(cross.mean()),
        taught=taught,
        mean_g=float(g.mean()),
        n_pot=int(pot.sum()),
        cross_frac_pot=float(cross[pot].mean()) if pot.any() else float("nan"),
        max_row_len=int(np.bincount(pre, minlength=N).max()))


def run_one(condition, seed, tsim_ms=None, verbose=False, settle_probes=4):
    """Build, run and measure one (condition, seed). Returns a metrics dict."""
    tsim_ms = TSIM_MS if tsim_ms is None else tsim_ms
    types = CONDITIONS[condition]

    t0 = time.time()
    model, ng_src, ng_tgt, sg_lat = build_model(seed)
    model.build()
    model.load()
    t_build = time.time() - t0

    steps_total = int(round(tsim_ms / DT))
    steps_per_block = int(round(PATTERN_BLOCK_MS / DT))
    steps_per_update = int(round(UPDATE_INTERVAL_MS / DT))
    probe_at = {int(steps_total * (k + 1) / settle_probes): k
                for k in range(settle_probes)}

    rates_var = ng_src.vars["stim_rates"]
    settle = []

    try:
        model.timestep = 0
        block = 0
        set_stim_rates(rates_var, types, block)

        for step in range(steps_total):
            model.step_time()
            if (step + 1) % steps_per_block == 0:
                block += 1
                set_stim_rates(rates_var, types, block)
            if (step + 1) % steps_per_update == 0:
                model.custom_update("UpdateConnectivity")
            if (step + 1) in probe_at:
                pre_p, post_p, _ = read_connectivity(sg_lat)
                settle.append(int((pre_p != post_p).sum()))

        pre, post, g = read_connectivity(sg_lat)
        sc_var = ng_tgt.vars["spikeCount"]
        sc_var.pull_from_device()
        spike_counts = np.asarray(sc_var.values, dtype=float)
    finally:
        model.unload()

    # Guard: if two probe points collapsed onto the same timestep (very short
    # runs) pad the front so every run reports the same number of probes and
    # the analysis cell can stack them into a rectangular array.
    while len(settle) < settle_probes:
        settle.insert(0, settle[0] if settle else 0)

    res = measure(pre, post, g, types)
    res["condition"] = condition
    res["seed"] = seed
    res["settle_edges"] = settle
    res["build_s"] = t_build
    res["total_s"] = time.time() - t0
    res["rate_hz_mean"] = float(spike_counts.mean() / (tsim_ms / 1000.0))
    res["rate_hz_max"] = float(spike_counts.max() / (tsim_ms / 1000.0))
    res["n_silent"] = int((spike_counts == 0).sum())

    if verbose:
        print("  {:<12} seed {:>3}: edges={:>6d} cross_frac={:.3f} "
              "rate={:.1f}Hz silent={:>3d} maxrow={:>3d} ({:.0f}s)".format(
                  condition, seed, res["n_edges"], res["cross_frac"],
                  res["rate_hz_mean"], res["n_silent"], res["max_row_len"],
                  res["total_s"]))
    return res

## 7. Smoke test - run this before the full sweepThis is where an untested notebook earns its keep. Run one short simulation percondition and check four things before spending an hour on the sweep:1. **It compiles and runs at all.** If not, everything below is moot.2. **Neurons fire.** `FF_WEIGHT` and `PEAK_RATE` are the two parameters in this   notebook that were *not* taken from a published source, and nothing guarantees   the values chosen make the LIF population spike. Target: mean rate in roughly   the 2-30 Hz range with few silent neurons. If the rate is ~0, raise   `FF_WEIGHT`. If it is pinned at the refractory ceiling (~200 Hz), lower it.3. **Synapses grow.** Edge count should be well above zero.4. **Row headroom.** `max_row_len` must stay clear of `MAX_ROW_LEN`; if it is   at the cap, synapses are being silently dropped and the numbers are invalid.Also: note the wall-clock time and multiply. Full sweep is`2 x len(SEEDS)` runs at `TSIM_MS / smoke_tsim` times this cost.

In [ ]:
SMOKE_TSIM_MS = 2000.0

smoke = []
for cond in ("segregated", "interleaved"):
    r = run_one(cond, seed=SEEDS[0], tsim_ms=SMOKE_TSIM_MS, verbose=True)
    smoke.append(r)

print()
for r in smoke:
    print("{:<12} edges={:<6d} autapses={:<4d} cross_frac={:.3f} "
          "mean_g={:.4f} potentiated={:<6d}".format(
              r["condition"], r["n_edges"], r["n_autapses"], r["cross_frac"],
              r["mean_g"], r["n_pot"]))
    print("             rates: mean={:.1f} Hz  max={:.1f} Hz  silent={}/{}".format(
        r["rate_hz_mean"], r["rate_hz_max"], r["n_silent"], N))
    print("             edge count over time (settling probes):", r["settle_edges"])
    print("             max row length {} / cap {}".format(r["max_row_len"], MAX_ROW_LEN))
    print("             build {:.0f}s, total {:.0f}s".format(r["build_s"], r["total_s"]))

est = sum(r["total_s"] for r in smoke) / 2.0 * (TSIM_MS / SMOKE_TSIM_MS) * 2 * len(SEEDS)
print("\nVery rough full-sweep estimate: {:.0f} min".format(est / 60.0))
print("(Build time does not scale with tsim, so this over-estimates somewhat.)")

## 8. Full sweep

In [ ]:
results = []
t_sweep = time.time()
for cond in ("segregated", "interleaved"):
    print("\n{}".format(cond.upper()))
    for s in SEEDS:
        results.append(run_one(cond, s, verbose=True))
print("\nsweep took {:.1f} min".format((time.time() - t_sweep) / 60.0))

## 9. ResultsDecision rule, lifted from `exp50_published_rule_confirmatory.py`:- **P1 fails** -> the run is COMPROMISED; do not interpret P2 until the parity or  settling failure is understood.- **P1 holds and P2a + P2b + P2c all hold** -> the segregation effect reproduces  on an independent engine under an independently-published rule.- **P1 holds, P2a holds, P2b and/or P2c fails** -> PARTIAL: the directional effect  is real at bit-identical geometry, but the "suppressed far below / saturated at  chance" calibration is off. Report the measured fractions, not the framing.- **P2a fails** -> the effect did NOT reproduce on GeNN at this power. That is a  publishable negative and must be reported as one.

In [ ]:
def col(rows, key):
    return np.array([r[key] for r in rows], dtype=float)


seg = [r for r in results if r["condition"] == "segregated"]
itl = [r for r in results if r["condition"] == "interleaved"]

print("=" * 84)
print("TOPOFORGE x GeNN: cross-type connectivity under a published, type-blind rule")
print("N = {}, NC = {}, {} seeds/condition, {} ms/run".format(
    N, NC, len(SEEDS), TSIM_MS))
print("=" * 84)

print("\n[P1b] manipulation check: did connectivity settle, and is there row headroom?")
p1b = True
for label, rows in (("segregated", seg), ("interleaved", itl)):
    e = col(rows, "n_edges")
    maxrow = col(rows, "max_row_len").max()
    probes = np.array([r["settle_edges"] for r in rows], dtype=float)
    # relative change over the final quarter of the run, averaged over seeds
    drift = np.abs(probes[:, -1] - probes[:, -2]) / np.maximum(probes[:, -1], 1.0)
    settled = bool(drift.mean() < SETTLE_TOL)
    nonzero = bool(e.mean() > 0)
    headroom = bool(maxrow < MAX_ROW_LEN)
    p1b = p1b and settled and nonzero and headroom
    print("  {:<12} edges {:>8.1f} +/- {:<7.1f} | last-quarter drift {:>6.1%} {} "
          "| max row {:>3.0f}/{} {}".format(
              label, e.mean(), e.std(ddof=1), drift.mean(),
              "OK" if settled else "NOT SETTLED",
              maxrow, MAX_ROW_LEN, "OK" if headroom else "AT CAP - INVALID"))
    print("  {:<12} mean edge count at 25/50/75/100% of run: {}".format(
        "", np.round(probes.mean(axis=0), 0).astype(int).tolist()))
    print("  {:<12} mean firing rate {:.1f} Hz, silent neurons {:.0f}/{}".format(
        "", col(rows, "rate_hz_mean").mean(), col(rows, "n_silent").mean(), N))
print("  P1b {}".format("HOLDS" if p1b else "FAILS"))

print("\n[P2] PRIMARY ENDPOINT: cross-type fraction of grown synapses")
cf_s, cf_i = col(seg, "cross_frac"), col(itl, "cross_frac")
t, p = stats.ttest_ind(cf_i, cf_s, equal_var=False)
pooled_sd = np.sqrt(((len(cf_i) - 1) * cf_i.var(ddof=1)
                     + (len(cf_s) - 1) * cf_s.var(ddof=1))
                    / (len(cf_i) + len(cf_s) - 2))
d = (cf_i.mean() - cf_s.mean()) / pooled_sd if pooled_sd > 0 else float("nan")

print("  segregated                     : {:.4f} +/- {:.4f}".format(
    cf_s.mean(), cf_s.std(ddof=1)))
print("  interleaved                    : {:.4f} +/- {:.4f}".format(
    cf_i.mean(), cf_i.std(ddof=1)))
print("  type-blind chance CEILING      : {:.4f}".format(CEILING))
print("  kernel-only analytic prediction: segregated {:.4f}, interleaved {:.4f}".format(
    kernel_cross_fraction(types_seg), kernel_cross_fraction(types_int)))
print("  Welch t = {:.2f}, p = {:.3e}, Cohen's d = {:.2f}".format(t, p, d))
print("  ratio interleaved/segregated   : {:.2f}x".format(
    cf_i.mean() / max(cf_s.mean(), 1e-12)))
print("  difference (int - seg)         : {:+.4f}".format(cf_i.mean() - cf_s.mean()))

p2a = bool(cf_i.mean() > cf_s.mean() and p < 0.05)
p2b = bool(cf_s.mean() < SEG_CEILING_MAX)
p2c = bool(cf_i.mean() > INT_CEILING_MIN)
print("\n  P2a  interleaved > segregated, p < 0.05            : {}".format(p2a))
print("  P2b  segregated  < {:.2f} (suppressed below chance)   : {}".format(
    SEG_CEILING_MAX, p2b))
print("  P2c  interleaved > {:.2f} (at/near the chance ceiling): {}".format(
    INT_CEILING_MIN, p2c))

print("\n[SECONDARY, reported not decisive]")
for name, key in (("cross-type fraction among potentiated synapses (g >= threshold)",
                   "cross_frac_pot"),
                  ("exp32b pattern-coupled pairs (0,3)+(1,4), raw count", "taught"),
                  ("mean synaptic weight g", "mean_g")):
    a, b = col(itl, key), col(seg, key)
    if np.isnan(a).all() or np.isnan(b).all():
        print("  {:<62} unavailable".format(name))
        continue
    tt, pp = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
    print("  {}".format(name))
    print("     segregated {:.4f} +/- {:.4f} | interleaved {:.4f} +/- {:.4f} "
          "| Welch p = {:.3e}".format(
              np.nanmean(b), np.nanstd(b, ddof=1),
              np.nanmean(a), np.nanstd(a, ddof=1), pp))

print("\n" + "=" * 84)
print("VERDICT")
print("=" * 84)
if not p1b:
    print("  COMPROMISED - the manipulation check failed. Do not interpret P2.")
elif p2a and p2b and p2c:
    print("  REPRODUCES on GeNN. Segregated placement suppresses cross-type")
    print("  connectivity far below the type-blind chance ceiling while interleaved")
    print("  sits at it, under a published rule implemented by a third party on an")
    print("  independent GPU simulator. This addresses the single-implementation")
    print("  limitation. It is NOT a hardware result.")
elif p2a:
    print("  PARTIAL. Direction and significance hold at bit-identical geometry,")
    print("  but the below-chance/at-chance calibration does not. Report the")
    print("  measured fractions, not the framing.")
else:
    print("  DID NOT REPRODUCE at this power. Report this as a negative result")
    print("  before assuming a bug - but check the P1b diagnostics first, since an")
    print("  unsettled or unsilenced network can mask a real effect.")

## 10. Reading the result honestly**The number to look at first is not the p-value, it is the comparison between theGeNN cross-type fraction and the kernel-only analytic prediction.** The Bamfordformation rule accepts a candidate synapse with probability`peak_prob * exp(-d^2 / 2*std^2)` and nothing else, so a large part of thesegregated-vs-interleaved gap is a direct consequence of the distance kernel andwould be there with the simulator switched off. That is not a flaw - exp50 hasexactly the same property, and it is *why* the endpoint is framed as suppressionbelow a chance ceiling rather than as a learning ratio. But it does bound whatthis notebook can claim:- It **can** show that an independent engine, running a third party's  implementation of a published rule, reproduces the placement-to-connectivity  effect at bit-identical geometry.- It **cannot** show that this rule learns better under interleaved placement.  The `cross_frac_pot` secondary endpoint is the closest thing here to an  activity-dependent measure, since it restricts to synapses STDP has driven above  the elimination threshold - but it is a secondary endpoint and it is not  pre-registered.**Things that would invalidate a positive result:**- `max_row_len` at `MAX_ROW_LEN` - synapses were silently dropped.- Near-zero firing rates - STDP never engaged, so `cross_frac_pot` is noise and  the result is pure geometry.- Unsettled edge counts - the comparison is between two transients, not two  steady states.- Segregated and interleaved edge *counts* differing a lot. The primary endpoint  is a fraction so this is normalised out of it, but a large asymmetry is worth  understanding rather than ignoring; exp50 saw one and said so.**A caveat specific to this geometry.** The reference's distance code usesperiodic boundary conditions, which are kept verbatim here. In the segregatedcondition that means the top stripe (type 4) wraps around and touches the bottomstripe (type 0), giving segregation one extra type boundary it would not have ona bounded sheet. This makes the segregated condition *less* extreme, so it isconservative with respect to the hypothesis.